# 07. Demo: Classifying Traffic Signs with the Final Model

**Project:** Road Sign Recognition Using Deep Learning (Cambodian Traffic Signs, 26 classes)

This notebook is a **demo only: nothing is trained or tuned**. It loads the final model selected in `05_hyperparameter_tuning.ipynb`, a fully fine-tuned ResNet-18 (Config B: learning rate 0.001, weight decay 0.001), and uses it to classify traffic-sign crops.

It shows, for each image:
- the image itself;
- the **predicted class** and its **confidence** (the model's softmax probability);
- the **top-3 predictions** with probabilities;
- for labelled test images, whether the prediction matches the **ground-truth label**.

> **Confidence is not a guarantee.** A confidence of 99% means the model strongly prefers that class, not that the prediction is certainly correct. On the 463-image test set the final model was wrong on 6 images, and some of those mistakes had confidence above 95%.

**Runtime:** Google Colab (a GPU is optional; the demo also runs on CPU). Run the cells from top to bottom. The presentation guide is at the end.

## 1. Setup

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

# ---- Edit these if your setup differs ------------------------------------
REPO_URL = "https://github.com/RothRothRoth/DeepLearning_CNN.git"
DRIVE_PROJECT_DIR = Path("/content/drive/MyDrive/Road_Sign_Recognition_CNN")
CHECKPOINT_RELATIVE = ("results/runs/hyperparameter_tuning/resnet18_finetuned_lr_wd_seed42/"
                       "final_tuned_model/resnet18_finetuned_tuned_best.pt")
# --------------------------------------------------------------------------

try:
    import google.colab  # noqa: F401

    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    PROJECT_ROOT = Path("/content/DeepLearning_CNN")
    if (PROJECT_ROOT / ".git").exists():
        subprocess.run(["git", "-C", str(PROJECT_ROOT), "pull", "--ff-only"], check=True)
    else:
        subprocess.run(["git", "clone", REPO_URL, str(PROJECT_ROOT)], check=True)
    PROCESSED_DIR = DRIVE_PROJECT_DIR / "dataset" / "processed"   # the verified test crops on Drive
    CHECKPOINT_PATH = DRIVE_PROJECT_DIR / CHECKPOINT_RELATIVE
else:
    # Local run: repository checkout + a downloaded copy of the checkpoint.
    PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
    PROCESSED_DIR = PROJECT_ROOT / "dataset" / "processed"
    CHECKPOINT_PATH = Path(os.environ.get("DEMO_CHECKPOINT", PROJECT_ROOT / "resnet18_finetuned_tuned_best.pt"))

os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Running in Colab: {IN_COLAB}")
print(f"Project root:     {PROJECT_ROOT}")
print(f"Test crops:       {PROCESSED_DIR}")
print(f"Checkpoint:       {CHECKPOINT_PATH}")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from PIL import Image

from src.config import CLASS_NAMES, IMAGE_SIZE, NUM_CLASSES
from src.data.dataset import resolve_crop_path
from src.data.transforms import get_eval_transforms
from src.models import create_resnet18_finetuned
from src.training import get_device, load_checkpoint

assert NUM_CLASSES == 26 and len(CLASS_NAMES) == 26
device = get_device()
print(f"PyTorch {torch.__version__} | device: {device}"
      + (f" ({torch.cuda.get_device_name(device)})" if device.type == "cuda" else ""))

## 2. Load the final trained model (no training)

The checkpoint was written by `save_checkpoint()` in `src/training/checkpoint.py` during the tuning run, and copied unchanged to `final_tuned_model/`. It is loaded the same way `05_hyperparameter_tuning.ipynb` loaded it for the final test evaluation:
1. `create_resnet18_finetuned(num_classes=26)` rebuilds the architecture: ResNet-18 with a `Linear(512, 26)` head. This first loads the ImageNet weights (a one-time download); they are then **replaced** by the trained weights.
2. `load_checkpoint(...)` loads the trained weights, with `weights_only=True`, and checks there are 26 classes.
3. `model.eval()` switches to inference mode: fixed BatchNorm statistics and no gradient updates.

In [ ]:
if not CHECKPOINT_PATH.is_file():
    raise FileNotFoundError(
        f"Final checkpoint not found at:\n  {CHECKPOINT_PATH}\n\n"
        "How to fix:\n"
        "  - Colab: make sure Google Drive is mounted with the account that holds the project, and that\n"
        f"    MyDrive/Road_Sign_Recognition_CNN/{CHECKPOINT_RELATIVE} exists\n"
        "    (it is created by notebooks/05_hyperparameter_tuning.ipynb).\n"
        "  - Local: download resnet18_finetuned_tuned_best.pt (link in the README) and set the\n"
        "    DEMO_CHECKPOINT environment variable to its path, or place it in the repository root."
    )

model = create_resnet18_finetuned(num_classes=NUM_CLASSES)
checkpoint = load_checkpoint(CHECKPOINT_PATH, model=model, map_location=device)
model.to(device).eval()

hp = checkpoint.get("hyperparameters", {})
print(f"Loaded checkpoint: {CHECKPOINT_PATH.name} ({CHECKPOINT_PATH.stat().st_size / 2**20:.1f} MiB)")
print(f"  model name:        {checkpoint.get('model_name')}")
print(f"  classes:           {checkpoint.get('num_classes')}")
print(f"  saved at epoch:    {checkpoint.get('epoch')}")
print(f"  best val accuracy: {checkpoint.get('best_metric', float('nan')):.4f} ({checkpoint.get('best_metric_name')})")
print(f"  learning rate:     {hp.get('learning_rate')} | weight decay: {hp.get('weight_decay')} | config: {hp.get('config')}")
print(f"  mode:              {'eval (inference)' if not model.training else 'TRAIN (unexpected)'}")
assert checkpoint.get("num_classes") == NUM_CLASSES
assert not model.training

### 2.1 Check: does the loaded checkpoint reproduce the saved test predictions?

The repository contains the final model's predictions for all 463 test images, saved by Notebook 05 (`final_tuned_model/test_predictions.csv`). This cell re-predicts a fixed sample of those images: all 6 misclassified ones plus 18 others. It checks that the predicted class and the confidence match. If they match, the checkpoint and preprocessing are exactly those used for the reported results.

This only re-uses the already-final model. Nothing is trained or selected here.

In [ ]:
EVAL_TRANSFORM = get_eval_transforms(IMAGE_SIZE)  # the exact val/test preprocessing (resize, tensor, ImageNet normalisation)


@torch.no_grad()
def predict(image, k=3):
    # Returns the top-k (class name, probability) pairs for one PIL image.
    x = EVAL_TRANSFORM(image.convert("RGB")).unsqueeze(0).to(device)
    probs = torch.softmax(model(x), dim=1)[0].cpu()
    top = torch.topk(probs, k)
    return [(CLASS_NAMES[i], float(p)) for p, i in zip(top.values, top.indices)]


def load_test_image(crop_path):
    return Image.open(resolve_crop_path(PROCESSED_DIR, crop_path)).convert("RGB")


saved = pd.read_csv(PROJECT_ROOT / "results/hyperparameter_tuning/resnet18_finetuned_lr_wd_seed42/"
                    "final_tuned_model/test_predictions.csv")
sample = pd.concat([saved[~saved["correct"]], saved[saved["correct"]].sample(18, random_state=42)])
mismatches = []
for _, row in sample.iterrows():
    (pred, conf), *_ = predict(load_test_image(row["crop_path"]))
    if pred != row["pred_label"] or abs(conf - row["confidence"]) > 1e-3:
        mismatches.append((row["crop_path"], row["pred_label"], round(row["confidence"], 4), pred, round(conf, 4)))

CHECKPOINT_VERIFIED = not mismatches
if CHECKPOINT_VERIFIED:
    print(f"PASS: all {len(sample)} re-predictions match the saved predictions (class and confidence).")
else:
    print(f"WARNING: {len(mismatches)} of {len(sample)} re-predictions differ from the saved results.")
    print("The loaded checkpoint or preprocessing is NOT the one behind the reported results. Examples:")
    for m in mismatches[:5]:
        print("  ", m)

## 3. Display helper

In [ ]:
def show_prediction(image, top3, true_label=None, title=None):
    # Shows the image, the top-3 probabilities and, if known, the ground truth.
    pred, conf = top3[0]
    fig, (ax_img, ax_bar) = plt.subplots(1, 2, figsize=(11, 4.2), gridspec_kw={"width_ratios": [1, 1.3]})
    ax_img.imshow(image)
    ax_img.axis("off")
    ax_img.set_title(title or "Input image", fontsize=11)

    names = [n for n, _ in top3][::-1]
    probs = [p for _, p in top3][::-1]
    colors = ["#DD8452" if n == pred else "#9DB8A9" for n in names]
    ax_bar.barh(names, probs, color=colors)
    for i, p in enumerate(probs):
        ax_bar.text(min(p + 0.02, 0.86), i, f"{100 * p:.2f}%", va="center", fontsize=10)
    ax_bar.set_xlim(0, 1)
    ax_bar.set_xlabel("Model confidence (softmax probability)")
    ax_bar.set_title("Top-3 predictions", fontsize=11)
    if true_label is not None:
        ok = pred == true_label
        fig.suptitle(f"Predicted {pred} ({100 * conf:.1f}%)  |  ground truth {true_label}  |  {'CORRECT' if ok else 'WRONG'}",
                     fontsize=12, fontweight="bold", color="#2E7D32" if ok else "#C62828")
    fig.tight_layout()
    plt.show()

    print(f"Predicted class: {pred}")
    print(f"Confidence:      {100 * conf:.2f}%  (the model's probability, not a guarantee of correctness)")
    print("Top-3:           " + " | ".join(f"{n} {100 * p:.2f}%" for n, p in top3))
    if true_label is not None:
        verdict = "CORRECT" if pred == true_label else "WRONG"
        print(f"Ground truth:    {true_label} -> prediction is {verdict}")

## 4. Demo A: a labelled image from the test set

Pick a class and an image number. The test images were never used for training or model selection, and their true labels are known, so the prediction can be checked.

In [ ]:
test_manifest = pd.read_csv(PROJECT_ROOT / "dataset/splits/test.csv")
print("Classes:", ", ".join(CLASS_NAMES))

CHOSEN_CLASS = "LEFT_BEND"   # any class name from the list above, or None for a random test image
IMAGE_NUMBER = 0             # which image of that class (0, 1, 2, ...)

if CHOSEN_CLASS is None:
    row = test_manifest.sample(1).iloc[0]
else:
    assert CHOSEN_CLASS in CLASS_NAMES, f"Unknown class {CHOSEN_CLASS!r}"
    rows = test_manifest[test_manifest["class_name"] == CHOSEN_CLASS].reset_index(drop=True)
    print(f"{CHOSEN_CLASS}: {len(rows)} test images (IMAGE_NUMBER 0 to {len(rows) - 1})")
    row = rows.iloc[IMAGE_NUMBER % len(rows)]

image = load_test_image(row["crop_path"])
show_prediction(image, predict(image), true_label=row["class_name"], title=row["crop_path"])

## 5. Demo B: one of the 6 test images the model got wrong

The final model misclassified 6 of 463 test images (`results/final_model/misclassified_test_images.csv`). Showing one keeps the demo honest:
- **Examples 0–2** appear to be annotation errors, based on visual inspection and comparison with the original dataset annotations. The model predicts the sign that is visible, but the label says otherwise.
- **Examples 3–5** are genuine model errors: a visually similar sign, a tiny blurred sign (39 × 39 px), and a sign covered by branches.

In [ ]:
misclassified = pd.read_csv(PROJECT_ROOT / "results/final_model/misclassified_test_images.csv")
print(misclassified[["true_label", "predicted_label", "confidence", "category"]].to_string())

EXAMPLE = 4   # 0-5, see the table above

row = misclassified.iloc[EXAMPLE]
image = load_test_image(row["crop_path"])
show_prediction(image, predict(image), true_label=row["true_label"],
                title=f"{row['category']}: {row['crop_path']}")
print(f"Note: {row['visual_observation']}")

## 6. Demo C: upload your own traffic-sign image

The model was trained on **tight crops of a single sign**. For a meaningful result, upload an image cropped closely around one sign. A full street photo, or a sign that is not one of the 26 classes, will still get a prediction, but it can be confidently wrong. The model has no "unknown" class.

There is no ground-truth label for an uploaded image, so correctness cannot be checked here.

In [ ]:
if IN_COLAB:
    from google.colab import files

    uploaded = files.upload()   # choose a .jpg / .png file
    for name, data in uploaded.items():
        import io

        image = Image.open(io.BytesIO(data)).convert("RGB")
        show_prediction(image, predict(image), title=f"Uploaded: {name}")
else:
    UPLOAD_PATH = None   # local run: set to an image path, e.g. "my_sign.jpg"
    if UPLOAD_PATH:
        image = Image.open(UPLOAD_PATH).convert("RGB")
        show_prediction(image, predict(image), title=f"Image: {UPLOAD_PATH}")
    else:
        print("Local run: set UPLOAD_PATH to an image file to try your own image.")

## How to present this demo (about 1–2 minutes)

**Before the presentation**
1. Open this notebook in Colab, run **all cells once** (Runtime → Run all), and keep the tab open. Loading the model takes a little time the first time.
2. Check that Section 2.1 prints **PASS**. It confirms you are demoing the exact model behind the reported results.
3. Optionally prepare one cropped sign image on your laptop for Demo C.

**During the presentation**
1. **Section 2:** "This loads my final model: a ResNet-18 fine-tuned on the training set, selected on validation data. No training happens here." Point at the printed config (lr 0.001, weight decay 0.001).
2. **Demo A:** change `CHOSEN_CLASS` (e.g. `"KEEP_RIGHT"`, `"NO_ENTRY"`) and re-run the cell. Read out the predicted class, the confidence and the ground truth.
3. **Demo B:** show a failure case. "Confidence is not certainty: here the model is wrong." Example 4 is the tiny 39-pixel sign. Examples 0–2 appear to be annotation errors in the dataset.
4. **Demo C (optional):** upload a cropped sign and explain the model always picks one of the 26 classes.

**If something goes wrong:** use screenshots of Demo A and B prepared in advance, or the error gallery in the slides.